In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import json
import time
import threading
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.impute import SimpleImputer
import joblib

# For real-time processing simulation
import queue
import logging
from collections import deque

class FraudDetectionSystem:
    """
    Complete AI-Powered Fraud Detection System
    Implements supervised and unsupervised learning for real-time fraud detection
    """

    def __init__(self):
        self.models = {}
        self.scalers = {}
        self.encoders = {}
        self.feature_columns = []
        self.alert_queue = queue.Queue()
        self.transaction_history = deque(maxlen=10000)
        self.metrics = {
            'total_transactions': 0,
            'fraud_detected': 0,
            'false_positives': 0,
            'accuracy': 0.0
        }
        self.setup_logging()

    def setup_logging(self):
        """Setup logging for the fraud detection system"""
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('fraud_detection.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)

    def generate_synthetic_data(self, n_samples: int = 10000, fraud_rate: float = 0.02) -> pd.DataFrame:
        """
        Generate synthetic transaction data for training
        """
        np.random.seed(42)

        # Generate normal transactions
        n_normal = int(n_samples * (1 - fraud_rate))
        n_fraud = n_samples - n_normal

        data = []

        # Normal transactions
        for i in range(n_normal):
            transaction = {
                'transaction_id': f'txn_{i}',
                'user_id': np.random.randint(1, 5000),
                'amount': np.random.lognormal(3, 1.5),  # Log-normal distribution
                'merchant_category': np.random.choice(['grocery', 'gas', 'restaurant', 'retail', 'online'],
                                                    p=[0.3, 0.2, 0.25, 0.15, 0.1]),
                'country': np.random.choice(['US', 'CA', 'UK', 'FR', 'DE'], p=[0.6, 0.15, 0.1, 0.08, 0.07]),
                'hour': np.random.normal(14, 4) % 24,  # Peak around 2 PM
                'day_of_week': np.random.randint(0, 7),
                'is_weekend': 0,
                'transaction_frequency': np.random.poisson(5),  # Transactions per day
                'device_risk_score': np.random.beta(2, 8),  # Low risk devices
                'is_fraud': 0
            }
            transaction['is_weekend'] = 1 if transaction['day_of_week'] >= 5 else 0
            data.append(transaction)

        # Fraudulent transactions - different patterns
        for i in range(n_fraud):
            transaction = {
                'transaction_id': f'fraud_txn_{i}',
                'user_id': np.random.randint(1, 5000),
                'amount': np.random.lognormal(5, 2),  # Higher amounts
                'merchant_category': np.random.choice(['online', 'cash_advance', 'unknown'],
                                                    p=[0.5, 0.3, 0.2]),
                'country': np.random.choice(['RU', 'NG', 'CN', 'US', 'UK'], p=[0.3, 0.2, 0.2, 0.2, 0.1]),
                'hour': np.random.choice([2, 3, 4, 22, 23, 0, 1], p=[0.15, 0.15, 0.15, 0.2, 0.15, 0.1, 0.1]),
                'day_of_week': np.random.randint(0, 7),
                'is_weekend': 0,
                'transaction_frequency': np.random.poisson(15),  # Higher frequency
                'device_risk_score': np.random.beta(8, 2),  # High risk devices
                'is_fraud': 1
            }
            transaction['is_weekend'] = 1 if transaction['day_of_week'] >= 5 else 0
            data.append(transaction)

        df = pd.DataFrame(data)

        # Add derived features
        df['amount_log'] = np.log1p(df['amount'])
        df['is_high_amount'] = (df['amount'] > df['amount'].quantile(0.95)).astype(int)
        df['is_unusual_hour'] = ((df['hour'] < 6) | (df['hour'] > 23)).astype(int)
        df['risk_score'] = (df['device_risk_score'] * 0.4 +
                           df['is_high_amount'] * 0.3 +
                           df['is_unusual_hour'] * 0.3)

        return df.sample(frac=1).reset_index(drop=True)  # Shuffle

    def preprocess_data(self, df: pd.DataFrame, is_training: bool = True) -> pd.DataFrame:
        """
        Preprocess the data for training/prediction
        """
        df_processed = df.copy()

        # Handle categorical variables
        categorical_columns = ['merchant_category', 'country']

        for col in categorical_columns:
            if is_training:
                le = LabelEncoder()
                df_processed[f'{col}_encoded'] = le.fit_transform(df_processed[col])
                self.encoders[col] = le
            else:
                if col in self.encoders:
                    # Handle unseen categories
                    try:
                        df_processed[f'{col}_encoded'] = self.encoders[col].transform(df_processed[col])
                    except ValueError:
                        # Assign unknown category to most common class
                        df_processed[f'{col}_encoded'] = 0

        # Select features
        feature_columns = [
            'amount', 'amount_log', 'hour', 'day_of_week', 'is_weekend',
            'transaction_frequency', 'device_risk_score', 'is_high_amount',
            'is_unusual_hour', 'risk_score', 'merchant_category_encoded', 'country_encoded'
        ]

        if is_training:
            self.feature_columns = feature_columns

        # Scale numerical features
        if is_training:
            scaler = StandardScaler()
            df_processed[feature_columns] = scaler.fit_transform(df_processed[feature_columns])
            self.scalers['standard'] = scaler
        else:
            if 'standard' in self.scalers:
                df_processed[feature_columns] = self.scalers['standard'].transform(df_processed[feature_columns])

        return df_processed

    def train_models(self, df: pd.DataFrame):
        """
        Train both supervised and unsupervised models
        """
        self.logger.info("Starting model training...")

        # Preprocess data
        df_processed = self.preprocess_data(df, is_training=True)

        X = df_processed[self.feature_columns]
        y = df_processed['is_fraud']

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        # 1. Supervised Learning - Random Forest
        self.logger.info("Training Random Forest classifier...")
        rf_model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        rf_model.fit(X_train, y_train)
        self.models['random_forest'] = rf_model

        # Evaluate supervised model
        rf_pred = rf_model.predict(X_test)
        rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]

        print("\n=== Random Forest Performance ===")
        print(classification_report(y_test, rf_pred))
        print(f"ROC-AUC Score: {roc_auc_score(y_test, rf_pred_proba):.4f}")

        # 2. Unsupervised Learning - Isolation Forest
        self.logger.info("Training Isolation Forest for anomaly detection...")

        # Train on normal transactions only
        X_normal = X_train[y_train == 0]
        iso_forest = IsolationForest(
            contamination=0.02,  # Expected fraud rate
            random_state=42,
            n_jobs=-1
        )
        iso_forest.fit(X_normal)
        self.models['isolation_forest'] = iso_forest

        # Evaluate anomaly detection
        iso_pred = iso_forest.predict(X_test)
        iso_pred_binary = (iso_pred == -1).astype(int)  # -1 indicates anomaly

        print("\n=== Isolation Forest Performance ===")
        print(classification_report(y_test, iso_pred_binary))

        # 3. Ensemble approach - combine both models
        self.logger.info("Creating ensemble model...")

        # Save feature importances
        feature_importance = pd.DataFrame({
            'feature': self.feature_columns,
            'importance': rf_model.feature_importances_
        }).sort_values('importance', ascending=False)

        print("\n=== Feature Importance ===")
        print(feature_importance.head(10))

        # Update metrics
        self.metrics['accuracy'] = rf_model.score(X_test, y_test)

        self.logger.info("Model training completed successfully!")

    def predict_fraud(self, transaction: Dict) -> Dict:
        """
        Predict fraud for a single transaction using ensemble approach
        """
        # Convert to DataFrame
        df = pd.DataFrame([transaction])

        # Preprocess
        df_processed = self.preprocess_data(df, is_training=False)
        X = df_processed[self.feature_columns]

        # Get predictions from both models
        rf_proba = 0.5  # Default
        iso_anomaly = 0.5  # Default

        if 'random_forest' in self.models:
            rf_proba = self.models['random_forest'].predict_proba(X)[0, 1]

        if 'isolation_forest' in self.models:
            iso_score = self.models['isolation_forest'].decision_function(X)[0]
            # Convert to probability-like score (0-1)
            iso_anomaly = max(0, min(1, (0.5 - iso_score) * 2))

        # Ensemble score (weighted average)
        ensemble_score = 0.7 * rf_proba + 0.3 * iso_anomaly

        # Decision logic
        if ensemble_score > 0.85:
            decision = 'BLOCKED'
            alert_level = 'CRITICAL'
        elif ensemble_score > 0.7:
            decision = 'REVIEW'
            alert_level = 'WARNING'
        else:
            decision = 'APPROVED'
            alert_level = 'INFO'

        result = {
            'transaction_id': transaction.get('transaction_id', 'unknown'),
            'risk_score': ensemble_score,
            'decision': decision,
            'alert_level': alert_level,
            'rf_score': rf_proba,
            'anomaly_score': iso_anomaly,
            'timestamp': datetime.now().isoformat(),
            'amount': transaction.get('amount', 0),
            'merchant_category': transaction.get('merchant_category', 'unknown')
        }

        return result

    def generate_real_time_transaction(self) -> Dict:
        """
        Generate a random transaction for real-time simulation
        """
        merchants = ['grocery', 'gas', 'restaurant', 'retail', 'online', 'cash_advance', 'unknown']
        countries = ['US', 'CA', 'UK', 'FR', 'RU', 'NG', 'CN']

        # Bias towards creating some suspicious transactions
        is_suspicious = np.random.random() < 0.1

        if is_suspicious:
            # Generate suspicious transaction
            transaction = {
                'transaction_id': f'txn_{int(time.time() * 1000)}',
                'user_id': np.random.randint(1, 5000),
                'amount': np.random.lognormal(5, 1.5),  # Higher amounts
                'merchant_category': np.random.choice(['online', 'cash_advance', 'unknown']),
                'country': np.random.choice(['RU', 'NG', 'CN']),
                'hour': np.random.choice([2, 3, 23, 0, 1]),
                'day_of_week': np.random.randint(0, 7),
                'transaction_frequency': np.random.poisson(20),
                'device_risk_score': np.random.beta(8, 2),
            }
        else:
            # Generate normal transaction
            transaction = {
                'transaction_id': f'txn_{int(time.time() * 1000)}',
                'user_id': np.random.randint(1, 5000),
                'amount': np.random.lognormal(3, 1),
                'merchant_category': np.random.choice(['grocery', 'gas', 'restaurant', 'retail']),
                'country': np.random.choice(['US', 'CA', 'UK']),
                'hour': np.random.normal(14, 4) % 24,
                'day_of_week': np.random.randint(0, 7),
                'transaction_frequency': np.random.poisson(5),
                'device_risk_score': np.random.beta(2, 8),
            }

        # Add derived features
        transaction['is_weekend'] = 1 if transaction['day_of_week'] >= 5 else 0
        transaction['amount_log'] = np.log1p(transaction['amount'])
        transaction['is_high_amount'] = 1 if transaction['amount'] > 1000 else 0
        transaction['is_unusual_hour'] = 1 if (transaction['hour'] < 6 or transaction['hour'] > 23) else 0
        transaction['risk_score'] = (transaction['device_risk_score'] * 0.4 +
                                   transaction['is_high_amount'] * 0.3 +
                                   transaction['is_unusual_hour'] * 0.3)

        return transaction

    def process_transaction_stream(self, duration_minutes: int = 5):
        """
        Simulate real-time transaction processing
        """
        self.logger.info(f"Starting real-time processing for {duration_minutes} minutes...")

        start_time = time.time()
        end_time = start_time + (duration_minutes * 60)

        while time.time() < end_time:
            # Generate new transaction
            transaction = self.generate_real_time_transaction()

            # Process transaction
            result = self.predict_fraud(transaction)

            # Update metrics
            self.metrics['total_transactions'] += 1
            if result['decision'] in ['BLOCKED', 'REVIEW']:
                self.metrics['fraud_detected'] += 1

            # Add to history
            self.transaction_history.append(result)

            # Generate alerts for high-risk transactions
            if result['alert_level'] in ['CRITICAL', 'WARNING']:
                alert = {
                    'timestamp': result['timestamp'],
                    'transaction_id': result['transaction_id'],
                    'message': f"{result['decision']}: ${result['amount']:.2f} transaction",
                    'risk_score': result['risk_score'],
                    'alert_level': result['alert_level']
                }
                self.alert_queue.put(alert)

            # Print real-time updates
            if self.metrics['total_transactions'] % 10 == 0:
                detection_rate = (self.metrics['fraud_detected'] / self.metrics['total_transactions']) * 100
                print(f"\nProcessed: {self.metrics['total_transactions']} | "
                      f"Fraud Detected: {self.metrics['fraud_detected']} | "
                      f"Detection Rate: {detection_rate:.1f}%")

                # Show recent high-risk transactions
                recent_high_risk = [t for t in list(self.transaction_history)[-5:]
                                  if t['decision'] in ['BLOCKED', 'REVIEW']]

                if recent_high_risk:
                    print("Recent High-Risk Transactions:")
                    for txn in recent_high_risk:
                        print(f"  {txn['decision']}: ${txn['amount']:.2f} | "
                              f"Risk: {txn['risk_score']:.3f} | "
                              f"ID: {txn['transaction_id']}")

            # Process alerts
            while not self.alert_queue.empty():
                try:
                    alert = self.alert_queue.get_nowait()
                    self.logger.warning(f"FRAUD ALERT: {alert['message']} | "
                                      f"Risk Score: {alert['risk_score']:.3f}")
                except queue.Empty:
                    break

            # Wait before next transaction (simulate real-world timing)
            time.sleep(np.random.exponential(2))  # Average 2 seconds between transactions

        self.logger.info("Real-time processing completed!")

    def generate_dashboard_report(self):
        """
        Generate a comprehensive dashboard report
        """
        print("\n" + "="*80)
        print("FRAUD DETECTION SYSTEM DASHBOARD")
        print("="*80)

        # Overall metrics
        detection_rate = (self.metrics['fraud_detected'] / max(1, self.metrics['total_transactions'])) * 100

        print(f"\n📊 SYSTEM METRICS:")
        print(f"Total Transactions Processed: {self.metrics['total_transactions']}")
        print(f"Fraudulent Transactions Detected: {self.metrics['fraud_detected']}")
        print(f"Detection Rate: {detection_rate:.2f}%")
        print(f"Model Accuracy: {self.metrics['accuracy']:.1f}%")

        # Recent transactions analysis
        if self.transaction_history:
            recent_transactions = list(self.transaction_history)[-20:]

            blocked = len([t for t in recent_transactions if t['decision'] == 'BLOCKED'])
            review = len([t for t in recent_transactions if t['decision'] == 'REVIEW'])
            approved = len([t for t in recent_transactions if t['decision'] == 'APPROVED'])

            print(f"\n🔍 RECENT ACTIVITY (Last 20 transactions):")
            print(f"Blocked: {blocked} | Under Review: {review} | Approved: {approved}")

            # Risk score distribution
            risk_scores = [t['risk_score'] for t in recent_transactions]
            avg_risk = np.mean(risk_scores)
            max_risk = max(risk_scores)

            print(f"\n⚠️  RISK ANALYSIS:")
            print(f"Average Risk Score: {avg_risk:.3f}")
            print(f"Maximum Risk Score: {max_risk:.3f}")

            # High-risk transactions
            high_risk = [t for t in recent_transactions if t['risk_score'] > 0.7]
            if high_risk:
                print(f"\n🚨 HIGH-RISK TRANSACTIONS:")
                for txn in high_risk[-5:]:  # Show last 5 high-risk
                    print(f"  ID: {txn['transaction_id'][:12]}... | "
                          f"${txn['amount']:.2f} | "
                          f"Risk: {txn['risk_score']:.3f} | "
                          f"Status: {txn['decision']}")

        print("\n" + "="*80)

    def save_models(self, filepath: str = 'fraud_models.joblib'):
        """Save trained models and preprocessors"""
        model_data = {
            'models': self.models,
            'scalers': self.scalers,
            'encoders': self.encoders,
            'feature_columns': self.feature_columns
        }
        joblib.dump(model_data, filepath)
        self.logger.info(f"Models saved to {filepath}")

    def load_models(self, filepath: str = 'fraud_models.joblib'):
        """Load trained models and preprocessors"""
        model_data = joblib.load(filepath)
        self.models = model_data['models']
        self.scalers = model_data['scalers']
        self.encoders = model_data['encoders']
        self.feature_columns = model_data['feature_columns']
        self.logger.info(f"Models loaded from {filepath}")


def main():
    """
    Main execution function - demonstrates the complete fraud detection pipeline
    """
    print("🚀 Initializing AI-Powered Fraud Detection System...")

    # Initialize system
    fraud_system = FraudDetectionSystem()

    # Step 1: Generate and prepare training data
    print("\n📊 Generating synthetic training data...")
    df = fraud_system.generate_synthetic_data(n_samples=50000, fraud_rate=0.02)
    print(f"Generated {len(df)} transactions ({df['is_fraud'].sum()} fraudulent)")

    # Step 2: Train models
    print("\n🤖 Training machine learning models...")
    fraud_system.train_models(df)

    # Step 3: Save models
    fraud_system.save_models()

    # Step 4: Test single transaction prediction
    print("\n🔍 Testing single transaction prediction...")
    test_transaction = {
        'transaction_id': 'test_001',
        'user_id': 12345,
        'amount': 5000.00,
        'merchant_category': 'online',
        'country': 'RU',
        'hour': 3,
        'day_of_week': 1,
        'is_weekend': 0,
        'transaction_frequency': 25,
        'device_risk_score': 0.95,
        'amount_log': np.log1p(5000.00),
        'is_high_amount': 1,
        'is_unusual_hour': 1,
        'risk_score': 0.85
    }

    result = fraud_system.predict_fraud(test_transaction)
    print(f"Test Result: {result['decision']} (Risk Score: {result['risk_score']:.3f})")

    # Step 5: Real-time processing simulation
    print("\n⚡ Starting real-time fraud detection simulation...")
    print("Processing transactions for 2 minutes...")

    # Run real-time processing in background
    fraud_system.process_transaction_stream(duration_minutes=2)

    # Step 6: Generate final report
    fraud_system.generate_dashboard_report()

    print("\n✅ Fraud Detection System Demo Completed!")
    print("\nKey Features Demonstrated:")
    print("  ✓ Synthetic data generation with realistic fraud patterns")
    print("  ✓ Feature engineering and preprocessing pipeline")
    print("  ✓ Supervised learning (Random Forest) for classification")
    print("  ✓ Unsupervised learning (Isolation Forest) for anomaly detection")
    print("  ✓ Ensemble modeling combining multiple approaches")
    print("  ✓ Real-time transaction processing and risk scoring")
    print("  ✓ Multi-threshold decision making (Block/Review/Approve)")
    print("  ✓ Real-time alerting system")
    print("  ✓ Comprehensive monitoring and reporting")
    print("  ✓ Model persistence and deployment readiness")


if __name__ == "__main__":
    main()

🚀 Initializing AI-Powered Fraud Detection System...

📊 Generating synthetic training data...
Generated 50000 transactions (1000 fraudulent)

🤖 Training machine learning models...

=== Random Forest Performance ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9800
           1       1.00      0.98      0.99       200

    accuracy                           1.00     10000
   macro avg       1.00      0.99      0.99     10000
weighted avg       1.00      1.00      1.00     10000

ROC-AUC Score: 1.0000

=== Isolation Forest Performance ===
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      9800
           1       0.44      0.81      0.57       200

    accuracy                           0.98     10000
   macro avg       0.72      0.89      0.78     10000
weighted avg       0.98      0.98      0.98     10000


=== Feature Importance ===
                      feature  importance
6 


Processed: 20 | Fraud Detected: 1 | Detection Rate: 5.0%
Recent High-Risk Transactions:
  BLOCKED: $944.66 | Risk: 0.979 | ID: txn_1759067206918



Processed: 30 | Fraud Detected: 2 | Detection Rate: 6.7%
Recent High-Risk Transactions:
  BLOCKED: $156.45 | Risk: 1.000 | ID: txn_1759067219823

Processed: 40 | Fraud Detected: 2 | Detection Rate: 5.0%

Processed: 50 | Fraud Detected: 2 | Detection Rate: 4.0%



Processed: 60 | Fraud Detected: 3 | Detection Rate: 5.0%
Recent High-Risk Transactions:
  BLOCKED: $210.27 | Risk: 1.000 | ID: txn_1759067279254

FRAUD DETECTION SYSTEM DASHBOARD

📊 SYSTEM METRICS:
Total Transactions Processed: 62
Fraudulent Transactions Detected: 3
Detection Rate: 4.84%
Model Accuracy: 1.0%

🔍 RECENT ACTIVITY (Last 20 transactions):
Blocked: 1 | Under Review: 0 | Approved: 19

⚠️  RISK ANALYSIS:
Average Risk Score: 0.235
Maximum Risk Score: 1.000

🚨 HIGH-RISK TRANSACTIONS:
  ID: txn_17590672... | $210.27 | Risk: 1.000 | Status: BLOCKED


✅ Fraud Detection System Demo Completed!

Key Features Demonstrated:
  ✓ Synthetic data generation with realistic fraud patterns
  ✓ Feature engineering and preprocessing pipeline
  ✓ Supervised learning (Random Forest) for classification
  ✓ Unsupervised learning (Isolation Forest) for anomaly detection
  ✓ Ensemble modeling combining multiple approaches
  ✓ Real-time transaction processing and risk scoring
  ✓ Multi-threshold decis